# Fine-tune YOLO11n from existing `best.pt` (4-class)

Combines:
- `barcode_best_training_images.zip` (DataMatrix)
- `barcode_dataset_merged.zip` (EAN13, CODE128, GS1_128)

Fine-tunes from existing `best.pt`.

## Target classes
- `0: datamatrix`
- `1: EAN13`
- `2: CODE128`
- `3: GS1_128`

## Inputs (Google Drive folder)
Folder: https://drive.google.com/drive/folders/1cDwbxWH3LJ0TLQpvP4EnZdynSKgFaCLj

Must contain:
1. `best.pt`
2. `barcode_best_training_images.zip`
3. `barcode_dataset_merged.zip`

> Runtime → Change runtime type → **T4 GPU**


## 1) GPU + install


In [ ]:
!nvidia-smi
!pip -q install -U ultralytics

import torch
import ultralytics
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("ultralytics", ultralytics.__version__)


## 2) Mount Google Drive + copy files to `/content`

Skips flaky `files.upload()` for large zips.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

FOLDER_ID = "1cDwbxWH3LJ0TLQpvP4EnZdynSKgFaCLj"
NEEDED = [
    "best.pt",
    "barcode_best_training_images.zip",
    "barcode_dataset_merged.zip",
]

# Prefer Colab shortcut-by-id path for shared/owned folders
candidates = [
    Path(f"/content/drive/.shortcut-targets-by-id/{FOLDER_ID}"),
    Path("/content/drive/MyDrive"),
]

folder = None
for c in candidates:
    if c.exists():
        folder = c
        break

assert folder is not None, "Drive mount failed / folder not found"

found = {}
for name in NEEDED:
    hits = list(folder.rglob(name)) if folder.name != "MyDrive" else list(Path("/content/drive").rglob(name))
    # If using MyDrive root, search whole Drive
    if folder.name == "MyDrive" or not hits:
        hits = list(Path("/content/drive").rglob(name))
    print(f"{name}: {len(hits)} hit(s)")
    for h in hits[:5]:
        print(" ", h)
    assert hits, f"Missing on Drive: {name}"
    found[name] = hits[0]

for name, src in found.items():
    dst = Path("/content") / name
    print(f"Copying {src} -> {dst}")
    shutil.copy2(src, dst)
    print("  OK", dst.stat().st_size)

print("All files ready in /content")


## 3) Extract datasets


In [ ]:
import zipfile
from pathlib import Path

dm_zip = Path("/content/barcode_best_training_images.zip")
bc_zip = Path("/content/barcode_dataset_merged.zip")
base_pt = Path("/content/best.pt")

assert dm_zip.exists(), f"Missing {dm_zip}"
assert bc_zip.exists(), f"Missing {bc_zip}"
assert base_pt.exists(), f"Missing {base_pt}"

dm_root = Path("/content/dm_ds")
bc_root = Path("/content/bc_ds")

for root, zpath in [(dm_root, dm_zip), (bc_root, bc_zip)]:
    if root.exists():
        import shutil
        shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)
    print("Extracting", zpath.name, "...")
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(root)

# Handle zip that unwraps into a nested folder named like barcode_dataset_merged
def unwrap_if_nested(root: Path) -> Path:
    # If root/images exists, good. Else find first **/images/train
    if (root / "images").exists():
        return root
    nested = list(root.rglob("images/train"))
    if nested:
        return nested[0].parent.parent
    return root

dm_root = unwrap_if_nested(dm_root)
bc_root = unwrap_if_nested(bc_root)

print("DM root:", dm_root, "images?", (dm_root / "images").exists())
print("BC root:", bc_root, "images?", (bc_root / "images").exists())

# Persist for later cells
Path("/content/DM_ROOT.txt").write_text(str(dm_root))
Path("/content/BC_ROOT.txt").write_text(str(bc_root))
print("OK")


## 4) Build combined 4-class dataset

Mapping:
- DataMatrix dataset: keep class `0`
- Barcode merged dataset: shift class ids by `+1`
  - `0→1` (EAN13), `1→2` (CODE128), `2→3` (GS1_128)


In [ ]:
import shutil
from collections import Counter
from pathlib import Path

dm_root = Path(Path("/content/DM_ROOT.txt").read_text().strip())
bc_root = Path(Path("/content/BC_ROOT.txt").read_text().strip())

combined = Path("/content/combined_4class")
if combined.exists():
    shutil.rmtree(combined)

for split in ["train", "val", "test"]:
    (combined / "images" / split).mkdir(parents=True, exist_ok=True)
    (combined / "labels" / split).mkdir(parents=True, exist_ok=True)

valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def copy_pair(src_img: Path, src_lbl: Path, dst_img_dir: Path, dst_lbl_dir: Path, class_shift: int, out_stem: str | None = None):
    if not src_img.exists() or not src_lbl.exists():
        return False
    stem = out_stem or src_img.stem
    dst_img = dst_img_dir / f"{stem}{src_img.suffix.lower()}"
    dst_lbl = dst_lbl_dir / f"{stem}.txt"
    shutil.copy2(src_img, dst_img)

    out_lines = []
    for line in src_lbl.read_text().strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cid = int(float(parts[0])) + class_shift
        out_lines.append(" ".join([str(cid)] + parts[1:]))
    dst_lbl.write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    return True

dm_counts = Counter()
for split in ["train", "val", "test"]:
    dm_img = dm_root / "images" / split
    dm_lbl = dm_root / "labels" / split
    if not dm_img.exists() or not dm_lbl.exists():
        continue
    for img in sorted(dm_img.iterdir()):
        if img.suffix.lower() not in valid_ext:
            continue
        lbl = dm_lbl / f"{img.stem}.txt"
        if copy_pair(img, lbl, combined / "images" / split, combined / "labels" / split, 0, out_stem=f"dm_{img.stem}"):
            dm_counts[split] += 1

bc_counts = Counter()
for split in ["train", "val", "test"]:
    bc_img = bc_root / "images" / split
    bc_lbl = bc_root / "labels" / split
    if not bc_img.exists() or not bc_lbl.exists():
        continue
    for img in sorted(bc_img.iterdir()):
        if img.suffix.lower() not in valid_ext:
            continue
        lbl = bc_lbl / f"{img.stem}.txt"
        if copy_pair(img, lbl, combined / "images" / split, combined / "labels" / split, 1, out_stem=f"bc_{img.stem}"):
            bc_counts[split] += 1

print("DM image counts:", dict(dm_counts))
print("BC image counts:", dict(bc_counts))
print("Combined root:", combined)
Path("/content/COMBINED_ROOT.txt").write_text(str(combined))


## 5) Write `data.yaml` + class counts


In [ ]:
from pathlib import Path
from collections import Counter

combined = Path(Path("/content/COMBINED_ROOT.txt").read_text().strip())

yaml_text = """path: /content/combined_4class
train: images/train
val: images/val
test: images/test

names:
  0: datamatrix
  1: EAN13
  2: CODE128
  3: GS1_128
"""
(combined / "data.yaml").write_text(yaml_text)
print((combined / "data.yaml").read_text())

img_counts = {}
box_counts = Counter()
for split in ["train", "val", "test"]:
    imgs = [p for p in (combined / "images" / split).iterdir() if p.is_file()]
    lbls = list((combined / "labels" / split).glob("*.txt"))
    img_counts[split] = len(imgs)
    for lf in lbls:
        for line in lf.read_text().strip().splitlines():
            line = line.strip()
            if line:
                box_counts[int(float(line.split()[0]))] += 1

print("Image counts:", img_counts)
print("Box counts by class:", dict(sorted(box_counts.items())))
assert sum(img_counts.values()) > 0, "No images in combined dataset"
assert (combined / "images" / "val").exists() and img_counts.get("val", 0) > 0, "val split empty"


## 6) Fine-tune from existing `best.pt`


In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("/content/best.pt")
DATA_YAML = Path("/content/combined_4class/data.yaml")
assert BEST_PT.exists() and DATA_YAML.exists()

PROJECT = "/content/runs/barcode"
RUN_NAME = "yolo11n_960_dm_ean_code128_gs1"
IMGSZ = 960
EPOCHS = 100
BATCH = 8  # drop to 4 if OOM

model = YOLO(str(BEST_PT))
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
)
print("Done. best.pt under", Path(PROJECT) / RUN_NAME / "weights")


## 7) Validate


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt — did training finish?"
BEST = candidates[0]
print("Using", BEST)

DATA_YAML = Path("/content/combined_4class/data.yaml")
model = YOLO(str(BEST))
metrics = model.val(data=str(DATA_YAML), imgsz=960)
print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Precision:", float(metrics.box.mp))
print("Recall:", float(metrics.box.mr))


## 8) Export ONNX (imgsz=960)


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
print("Exporting", BEST)

model = YOLO(str(BEST))
out = model.export(format="onnx", imgsz=960, simplify=True)
print("ONNX:", out)

onnx_src = Path(out)
onnx_dst = Path("/content/barcode-yolo11n-4class.onnx")
onnx_dst.write_bytes(onnx_src.read_bytes())
print("Wrote", onnx_dst, onnx_dst.stat().st_size // 1024, "KB")


## 9) Download artifacts

After download, locally you can keep them under:
```bash
mkdir -p runs/barcode/yolo11n_960_dm_ean_code128_gs1/weights
cp ~/Downloads/best.pt runs/barcode/yolo11n_960_dm_ean_code128_gs1/weights/
cp ~/Downloads/barcode-yolo11n-4class.onnx public/models/
```

> Note: app currently expects single-class ONNX (`1×5×N`). 4-class model needs parser update before replacing `public/models/barcode-yolo11n.onnx`.


In [ ]:
from google.colab import files
from pathlib import Path

PROJECT = Path("/content/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
ONNX = Path("/content/barcode-yolo11n-4class.onnx")
if not ONNX.exists():
    siblings = list(BEST.parent.glob("*.onnx"))
    assert siblings, "Run export cell first"
    ONNX = siblings[0]

print("Downloading", BEST)
files.download(str(BEST))
print("Downloading", ONNX)
files.download(str(ONNX))


## Notes
- **OOM:** set `BATCH = 4`
- **Drive files not found:** Add folder to My Drive (shortcut), re-run cell 2
- **Runtime restart:** remount Drive, re-run from cell 2
- Optional: save run to Drive:
```python
!cp -r /content/runs/barcode/yolo11n_960_dm_ean_code128_gs1 /content/drive/MyDrive/
```
